In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, f1_score, precision_score, recall_score)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")

# Create directories
import os
os.makedirs('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/models', exist_ok=True)

# ============================================================
# LOAD PROCESSED DATA
# ============================================================

df = pd.read_pickle('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed/hc2024_processed.pkl')

print("="*80)
print("PREDICTIVE MODELING — NAMCS HC 2024")
print("="*80)
print(f"Dataset: {len(df):,} visits\n")

# ============================================================
# MODEL 1: CHRONIC DISEASE CLASSIFIER (Multi-Label)
# ============================================================

print("="*80)
print("MODEL 1: CHRONIC DISEASE PREVALENCE CLASSIFIER")
print("="*80)
print("Objective: Predict presence of diabetes, hypertension, and mental health disorders\n")

# Prepare features
feature_cols = ['AGE', 'sex_binary', 'num_diagnoses', 'is_weekend']

# Add race/ethnicity dummies
race_dummies = pd.get_dummies(df['race_simple'], prefix='race', drop_first=True)
season_dummies = pd.get_dummies(df['season'], prefix='season', drop_first=True)

# Combine features
X_chronic = pd.concat([
    df[feature_cols],
    race_dummies,
    season_dummies
], axis=1)

# Handle missing values
X_chronic = X_chronic.fillna(X_chronic.mean())

# Targets (top 3 chronic conditions)
targets = ['diabetes', 'hypertension', 'mental_health']
y_chronic = df[targets]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_chronic, y_chronic, test_size=0.25, random_state=42
)

print(f"Training set: {len(X_train):,} visits")
print(f"Test set: {len(X_test):,} visits")
print(f"Features: {list(X_chronic.columns)}\n")

# Train individual classifiers for each condition
results_chronic = {}

for condition in targets:
    print(f"\nTraining classifier for: {condition}")
    print("-" * 60)
    
    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=100,
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train[condition])
    y_pred = rf.predict(X_test)
    y_pred_proba = rf.predict_proba(X_test)[:, 1]
    
    # Metrics
    f1 = f1_score(y_test[condition], y_pred)
    precision = precision_score(y_test[condition], y_pred)
    recall = recall_score(y_test[condition], y_pred)
    auc = roc_auc_score(y_test[condition], y_pred_proba)
    
    results_chronic[condition] = {
        'model': rf,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc,
        'y_test': y_test[condition],
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  F1 Score: {f1:.3f}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall: {recall:.3f}")
    print(f"  AUC-ROC: {auc:.3f}")
    
    # Feature importance
    feat_imp = pd.DataFrame({
        'feature': X_chronic.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False).head(5)
    
    print(f"\n  Top 5 Features:")
    for idx, row in feat_imp.iterrows():
        print(f"    {row['feature']:20s}: {row['importance']:.4f}")
    
    # Save model
    model_path = f'/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/models/{condition}_classifier.pkl'
    joblib.dump(rf, model_path)
    print(f"\n  ✓ Saved: models/{condition}_classifier.pkl")

# Summary table
print("\n" + "="*80)
print("MODEL 1 PERFORMANCE SUMMARY")
print("="*80)

model1_summary = pd.DataFrame({
    'Condition': targets,
    'F1 Score': [results_chronic[c]['f1'] for c in targets],
    'Precision': [results_chronic[c]['precision'] for c in targets],
    'Recall': [results_chronic[c]['recall'] for c in targets],
    'AUC-ROC': [results_chronic[c]['auc'] for c in targets]
})

print(model1_summary.to_string(index=False))

# ============================================================
# MODEL 2: HIGH COMPLEXITY VISIT PREDICTOR
# ============================================================

print("\n" + "="*80)
print("MODEL 2: HIGH COMPLEXITY VISIT PREDICTOR")
print("="*80)
print("Objective: Predict visits with 3+ diagnoses (high complexity)\n")

# Prepare features (use chronic disease flags as features)
feature_cols_complexity = ['AGE', 'sex_binary', 'diabetes', 'hypertension', 
                           'mental_health', 'obesity', 'substance_use', 'is_weekend']

X_complexity = pd.concat([
    df[feature_cols_complexity],
    race_dummies,
    season_dummies
], axis=1)

X_complexity = X_complexity.fillna(X_complexity.mean())

# Target
y_complexity = df['high_complexity']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_complexity, y_complexity, test_size=0.25, random_state=42, stratify=y_complexity
)

print(f"Training set: {len(X_train):,} visits")
print(f"Test set: {len(X_test):,} visits")
print(f"High complexity rate: {y_train.mean()*100:.1f}%\n")

# Gradient Boosting Classifier
print("Training Gradient Boosting Classifier...")

gbc = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

gbc.fit(X_train, y_train)

# Predictions
y_pred_complexity = gbc.predict(X_test)
y_pred_proba_complexity = gbc.predict_proba(X_test)[:, 1]

# Metrics
f1_complexity = f1_score(y_test, y_pred_complexity)
precision_complexity = precision_score(y_test, y_pred_complexity)
recall_complexity = recall_score(y_test, y_pred_complexity)
auc_complexity = roc_auc_score(y_test, y_pred_proba_complexity)

print(f"\n{'='*80}")
print("MODEL 2 PERFORMANCE")
print("="*80)
print(f"F1 Score: {f1_complexity:.3f}")
print(f"Precision: {precision_complexity:.3f}")
print(f"Recall: {recall_complexity:.3f}")
print(f"AUC-ROC: {auc_complexity:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_complexity, 
                          target_names=['Low Complexity', 'High Complexity']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_complexity)
print(cm)

# Feature importance
feat_imp_complexity = pd.DataFrame({
    'feature': X_complexity.columns,
    'importance': gbc.feature_importances_
}).sort_values('importance', ascending=False).head(10)

print("\nTop 10 Most Important Features:")
print(feat_imp_complexity.to_string(index=False))

# Save model
joblib.dump(gbc, '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/models/high_complexity_predictor.pkl')
print("\n✓ Saved: models/high_complexity_predictor.pkl")

# ============================================================
# MODEL 3: MULTIMORBIDITY PREDICTOR
# ============================================================

print("\n" + "="*80)
print("MODEL 3: MULTIMORBIDITY PREDICTOR")
print("="*80)
print("Objective: Predict patients with 2+ chronic conditions\n")

# Prepare features (exclude chronic disease flags to avoid data leakage)
feature_cols_multi = ['AGE', 'sex_binary', 'num_diagnoses', 'is_weekend', 'preventive_visit']

X_multi = pd.concat([
    df[feature_cols_multi],
    race_dummies,
    season_dummies
], axis=1)

X_multi = X_multi.fillna(X_multi.mean())

# Target
y_multi = df['multimorbidity']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.25, random_state=42, stratify=y_multi
)

print(f"Training set: {len(X_train):,} visits")
print(f"Test set: {len(X_test):,} visits")
print(f"Multimorbidity rate: {y_train.mean()*100:.1f}%\n")

# Logistic Regression with scaling
print("Training Logistic Regression Classifier...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

lr.fit(X_train_scaled, y_train)

# Predictions
y_pred_multi = lr.predict(X_test_scaled)
y_pred_proba_multi = lr.predict_proba(X_test_scaled)[:, 1]

# Metrics
f1_multi = f1_score(y_test, y_pred_multi)
precision_multi = precision_score(y_test, y_pred_multi)
recall_multi = recall_score(y_test, y_pred_multi)
auc_multi = roc_auc_score(y_test, y_pred_proba_multi)

print(f"\n{'='*80}")
print("MODEL 3 PERFORMANCE")
print("="*80)
print(f"F1 Score: {f1_multi:.3f}")
print(f"Precision: {precision_multi:.3f}")
print(f"Recall: {recall_multi:.3f}")
print(f"AUC-ROC: {auc_multi:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_multi, 
                          target_names=['No Multimorbidity', 'Multimorbidity']))

print("\nConfusion Matrix:")
cm_multi = confusion_matrix(y_test, y_pred_multi)
print(cm_multi)

# Feature importance (coefficients)
feat_imp_multi = pd.DataFrame({
    'feature': X_multi.columns,
    'coefficient': np.abs(lr.coef_[0])
}).sort_values('coefficient', ascending=False).head(10)

print("\nTop 10 Most Important Features:")
print(feat_imp_multi.to_string(index=False))

# Save model and scaler
joblib.dump(lr, '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/models/multimorbidity_predictor.pkl')
joblib.dump(scaler, '/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/models/multimorbidity_scaler.pkl')
print("\n✓ Saved: models/multimorbidity_predictor.pkl")
print("✓ Saved: models/multimorbidity_scaler.pkl")

# ============================================================
# MASTER PERFORMANCE SUMMARY
# ============================================================

print("\n" + "="*80)
print("MASTER MODEL PERFORMANCE SUMMARY")
print("="*80)

master_summary = pd.DataFrame({
    'Model': [
        'Diabetes Classifier',
        'Hypertension Classifier',
        'Mental Health Classifier',
        'High Complexity Predictor',
        'Multimorbidity Predictor'
    ],
    'Algorithm': [
        'Random Forest',
        'Random Forest',
        'Random Forest',
        'Gradient Boosting',
        'Logistic Regression'
    ],
    'F1 Score': [
        results_chronic['diabetes']['f1'],
        results_chronic['hypertension']['f1'],
        results_chronic['mental_health']['f1'],
        f1_complexity,
        f1_multi
    ],
    'AUC-ROC': [
        results_chronic['diabetes']['auc'],
        results_chronic['hypertension']['auc'],
        results_chronic['mental_health']['auc'],
        auc_complexity,
        auc_multi
    ],
    'Business Impact': [
        'Early intervention targeting',
        'Blood pressure management',
        'Mental health screening',
        'Resource allocation planning',
        'Care coordination priority'
    ]
})

print("\n" + master_summary.to_string(index=False))

# Save summary
master_summary.to_csv('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/outputs/model_performance_summary.csv', 
                      index=False)
print("\n✓ Saved: outputs/model_performance_summary.csv")

# ============================================================
# VISUALIZATION: ROC CURVES
# ============================================================

print("\nCreating ROC Curve Visualization...")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Plot 1-3: Chronic disease classifiers
for i, condition in enumerate(targets):
    ax = axes[i]
    
    fpr, tpr, _ = roc_curve(results_chronic[condition]['y_test'], 
                            results_chronic[condition]['y_pred_proba'])
    auc_val = results_chronic[condition]['auc']
    
    ax.plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_val:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=10, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=10, fontweight='bold')
    ax.set_title(f'{condition.replace("_", " ").title()} Classifier', 
                 fontsize=11, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

# Plot 4: High complexity
ax = axes[3]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_complexity)
ax.plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_complexity:.3f}')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=10, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=10, fontweight='bold')
ax.set_title('High Complexity Predictor', fontsize=11, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Plot 5: Multimorbidity
ax = axes[4]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_multi)
ax.plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_multi:.3f}')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=10, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=10, fontweight='bold')
ax.set_title('Multimorbidity Predictor', fontsize=11, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Hide last subplot
axes[5].axis('off')

plt.suptitle('Model Performance: ROC Curves', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures/05_roc_curves.png', 
            dpi=300, bbox_inches='tight')
print("✓ Saved: figures/05_roc_curves.png")
plt.close()

print("\n" + "="*80)
print("PREDICTIVE MODELING COMPLETE")
print("="*80)
print("✓ Trained 5 models (3 chronic disease + complexity + multimorbidity)")
print("✓ Saved all models to models/ directory")
print("✓ Generated performance summary and ROC curves")
print("✓ Ready for business impact analysis")

PREDICTIVE MODELING — NAMCS HC 2024
Dataset: 503,799 visits

MODEL 1: CHRONIC DISEASE PREVALENCE CLASSIFIER
Objective: Predict presence of diabetes, hypertension, and mental health disorders

Training set: 377,849 visits
Test set: 125,950 visits
Features: ['AGE', 'sex_binary', 'num_diagnoses', 'is_weekend', 'race_Hispanic', 'race_Other', 'race_White', 'season_Spring', 'season_Summer', 'season_Winter']


Training classifier for: diabetes
------------------------------------------------------------
  F1 Score: 0.126
  Precision: 0.582
  Recall: 0.071
  AUC-ROC: 0.884

  Top 5 Features:
    num_diagnoses       : 0.5442
    AGE                 : 0.4128
    race_White          : 0.0197
    sex_binary          : 0.0105
    race_Hispanic       : 0.0058

  ✓ Saved: models/diabetes_classifier.pkl

Training classifier for: hypertension
------------------------------------------------------------
  F1 Score: 0.616
  Precision: 0.680
  Recall: 0.563
  AUC-ROC: 0.927

  Top 5 Features:
    num_diag